# Netflix Monthly SWOT

## 0. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re, gc, json, glob
import numpy as np
import pandas as pd
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

SWOT_LABELS = ["Strength", "Weakness", "Opportunity", "Threat"]
UNCLASSIFIED = "Unclassified"

print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())

CUDA available: True | device count: 2


In [2]:
!pip install -q langdetect emoji rapidfuzz bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 41.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.1 MB/s eta 0:00:00


## 1. Inference Parameters

In [3]:
import json
from pathlib import Path

MODEL_PATH = Path(
    "/kaggle/input/datasets/anastasiapolitidou/"
    "netflix-swot-model/swot_roberta_base_regularized/best_model_verified"
)

assert MODEL_PATH.is_dir(), f"Model directory not found: {MODEL_PATH}"

with open(MODEL_PATH / "inference_params.json", encoding="utf-8") as f:
    INFERENCE_PARAMS = json.load(f)

with open(MODEL_PATH / "test_metrics.json", encoding="utf-8") as f:
    TEST_METRICS = json.load(f)

assert INFERENCE_PARAMS["prediction_rule"] == "softmax_argmax"
assert INFERENCE_PARAMS["problem_type"] == "single_label_classification"
assert INFERENCE_PARAMS["swot_labels"] == SWOT_LABELS

MAX_LENGTH = int(INFERENCE_PARAMS["max_length"])
MACRO_F1 = float(TEST_METRICS["macro_f1"])
TEST_ACCURACY = float(TEST_METRICS["accuracy"])

print("Model:", MODEL_PATH)
print("Classification:", INFERENCE_PARAMS["prediction_rule"])
print("SWOT labels:", INFERENCE_PARAMS["swot_labels"])
print("Maximum tokens:", MAX_LENGTH)
print(f"Test Accuracy: {TEST_ACCURACY:.4f}")
print(f"Test Macro F1: {MACRO_F1:.4f}")

Model: /kaggle/input/datasets/anastasiapolitidou/netflix-swot-model/swot_roberta_base_regularized/best_model_verified
Classification: softmax_argmax
SWOT labels: ['Strength', 'Weakness', 'Opportunity', 'Threat']
Maximum tokens: 256
Test Accuracy: 0.8500
Test Macro F1: 0.7882


## 2. Latest Google Play Reviews

In [4]:
import kagglehub

_ds = kagglehub.dataset_download(
    "ashishkumarak/netflix-reviews-playstore-daily-updated"
)

_csvs = sorted(glob.glob(os.path.join(_ds, "*.csv")))
assert len(_csvs) == 1, f"Expected one CSV, found: {_csvs}"

REVIEWS_PATH = _csvs[0]

df_raw = pd.read_csv(REVIEWS_PATH)
df_raw = df_raw[["content", "score", "at"]].rename(
    columns={"content": "text", "score": "rating", "at": "date"}
)

df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["rating"] = pd.to_numeric(df_raw["rating"], errors="coerce")

CURRENT_MONTH = pd.Timestamp.now(tz="UTC").tz_localize(None).to_period("M") - 1
PREVIOUS_MONTH = CURRENT_MONTH - 1
REPORTING_END = (CURRENT_MONTH + 1).to_timestamp()
df_raw = df_raw.loc[
    df_raw["date"].ge(pd.Timestamp("2020-01-01")) & df_raw["date"].lt(REPORTING_END)
].reset_index(drop=True)

review_month = df_raw["date"].dt.to_period("M")

df_current_month = df_raw[
    review_month == CURRENT_MONTH
].copy()

df_previous_month = df_raw[
    review_month == PREVIOUS_MONTH
].copy()

print("Dataset:", REVIEWS_PATH)
print("Most recent review:", df_raw["date"].max())
print("Total historical reviews:", len(df_raw))
print()
print("Current reporting month:", CURRENT_MONTH)
print("Reviews:", len(df_current_month))
print()
print("Previous reporting month:", PREVIOUS_MONTH)
print("Reviews:", len(df_previous_month))

assert len(df_current_month) > 0, (
    f"No reviews available for {CURRENT_MONTH}. "
    "Check whether the source dataset has been updated."
)

Dataset: /kaggle/input/datasets/ashishkumarak/netflix-reviews-playstore-daily-updated/netflix_reviews.csv
Most recent review: 2026-10-31 08:21:27
Total historical reviews: 142474

Current reporting month: 2026-10
Reviews: 3022

Previous reporting month: 2026-09
Reviews: 3707


## 3. Review Cleaning

In [5]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
import emoji
from rapidfuzz import process, fuzz

DetectorFactory.seed = SEED

LOW_INFO = {
    "good", "bad", "nice", "ok", "wow", "love it",
    "hate it", "very good", "very bad", "awesome", "terrible"
}

SPAM_PATTERNS = [
    r"earn money", r"work from home", r"visit my channel",
    r"check out my", r"promo code", r"free followers",
    r"telegram", r"whatsapp"
]

def detect_lang_safe(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def clean_text(text):
    text = str(text).strip().lower()
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s\.\,\!\?\-\'\"]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_low_quality(text):
    if not isinstance(text, str):
        return True
    txt = text.strip()
    if len(txt) < 20 or len(txt.split()) < 4:
        return True
    if txt.lower() in LOW_INFO:
        return True
    if re.search(r"(.)\1{5,}", txt):
        return True
    return False

def is_spam_like(text):
    txt = text.lower()
    return any(re.search(p, txt) for p in SPAM_PATTERNS)

def remove_near_duplicates(frame, text_col="text_clean", threshold=97):
    frame = frame.reset_index(drop=True)
    block = frame[text_col].str[:20]
    drop = set()

    for _, g in frame.groupby(block):
        if len(g) < 2:
            continue

        texts = g[text_col].tolist()
        sim = process.cdist(
            texts, texts, scorer=fuzz.ratio, workers=-1
        )
        gidx = g.index.to_numpy()

        for i in range(len(texts)):
            if gidx[i] in drop:
                continue

            dup = np.where(sim[i, i + 1:] >= threshold)[0] + i + 1
            drop.update(gidx[j] for j in dup)

    return frame.drop(index=list(drop)).reset_index(drop=True)


df = df_raw.dropna(subset=["text", "date"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() >= 15].copy()

df["lang"] = df["text"].apply(detect_lang_safe)
df = df[df["lang"] == "en"].copy()

english_count = len(df)

df["text_clean"] = df["text"].apply(clean_text)
df = df[~df["text_clean"].apply(is_low_quality)]
df = df[~df["text_clean"].apply(is_spam_like)]

df["review_month"] = df["date"].dt.to_period("M")

df = df.drop_duplicates(
    subset=["review_month", "text_clean"]
).reset_index(drop=True)

monthly_frames = [
    remove_near_duplicates(group, threshold=97)
    for _, group in df.groupby("review_month", sort=False)
]

df = (
    pd.concat(monthly_frames, ignore_index=True)
    if monthly_frames else df.iloc[:0].copy()
)

df = df.reset_index(drop=True)
df["review_id"] = np.arange(len(df))

df_current_clean = df[
    df["review_month"] == CURRENT_MONTH
].copy()

df_previous_clean = df[
    df["review_month"] == PREVIOUS_MONTH
].copy()

monthly_summary = pd.DataFrame([
    {
        "Month": str(month),
        "Raw reviews": int(
            (df_raw["date"].dt.to_period("M") == month).sum()
        ),
        "Cleaned English reviews": int(
            (df["review_month"] == month).sum()
        )
    }
    for month in [PREVIOUS_MONTH, CURRENT_MONTH]
])

print("Initial historical reviews:", len(df_raw))
print("English reviews before cleaning:", english_count)
print("Final cleaned reviews:", len(df))
print()
display(monthly_summary)

Initial historical reviews: 142474
English reviews before cleaning: 120613
Final cleaned reviews: 116116



,Month,Raw reviews,Cleaned English reviews
0,2026-09,3707,1966
1,2026-10,3022,1537


## 4. Opinion Segmentation

In [6]:
import spacy

nlp = spacy.load(
    "en_core_web_sm",
    disable=["tagger", "attribute_ruler", "lemmatizer", "ner"]
)

MIN_TOKENS = 4
MIN_PART_TOKENS = 4

CONTRAST = re.compile(r"\b(?:but|however)\b", re.I)

_LEAD_CONNECT = re.compile(
    r"^(?:and|or|so|also|then|plus|because|but|however)(?=\s|,|$)[\s,]*",
    re.I
)

_SUBJECTLESS = re.compile(
    r"^(?:are|is|was|were|am|be|been|being|has|have|had)\b",
    re.I
)

_DEPENDENT = re.compile(
    r"^(?:no matter|even if|even though|whether|as long as|in case)\b",
    re.I
)

def _tidy(seg):
    seg = _LEAD_CONNECT.sub("", str(seg).strip(" ,.-"))
    return seg.strip(" ,.-")

def _valid(seg, min_tok):
    toks = seg.split()

    if len(toks) < min_tok:
        return False
    if _SUBJECTLESS.match(seg):
        return False
    if _DEPENDENT.match(seg) and len(toks) < 12:
        return False

    return True

def split_doc_into_segments(doc):
    segments = []

    for sent in doc.sents:
        s = _tidy(sent.text)

        if not s:
            continue

        parts = [_tidy(p) for p in CONTRAST.split(s)]
        parts = [p for p in parts if p]

        if len(parts) >= 2 and all(
            _valid(p, MIN_PART_TOKENS) for p in parts
        ):
            segments.extend(parts)
        elif _valid(s, MIN_TOKENS):
            segments.append(s)

    return segments

def split_review_into_segments(text):
    return split_doc_into_segments(nlp(str(text)))


segment_rows = []

for row, doc in zip(
    df.itertuples(index=False),
    nlp.pipe(df["text_clean"].astype(str), batch_size=256)
):
    for seg_id, seg in enumerate(split_doc_into_segments(doc)):
        segment_rows.append({
            "review_id": row.review_id,
            "segment_id": seg_id,
            "text_clean": seg,
            "rating": row.rating,
            "date": row.date
        })

segments_df = pd.DataFrame(
    segment_rows,
    columns=["review_id", "segment_id", "text_clean", "rating", "date"]
)

assert not segments_df.empty, "No segments generated."

print("Total opinion segments:", len(segments_df))
print("Source reviews:", segments_df["review_id"].nunique())

monthly_segments = pd.DataFrame([
    {
        "Month": str(month),
        "Cleaned reviews": len(
            df[df["review_month"] == month]
        ),
        "Reviews with segments": segments_df.loc[
            segments_df["date"].dt.to_period("M") == month,
            "review_id"
        ].nunique(),
        "Opinion segments": int(
            (segments_df["date"].dt.to_period("M") == month).sum()
        )
    }
    for month in [PREVIOUS_MONTH, CURRENT_MONTH]
])

display(monthly_segments)
display(segments_df.head(10))

Total opinion segments: 252290
Source reviews: 115183


,Month,Cleaned reviews,Reviews with segments,Opinion segments
0,2026-09,1966,1930,3342
1,2026-10,1537,1499,2495


,review_id,segment_id,text_clean,rating,date
0,0,0,why is netflix stealing my money without my co...,1,2026-10-31 08:21:27
1,0,1,they are just deducting money from my bank car...,1,2026-10-31 08:21:27
2,0,2,this has happened to me the second time being ...,1,2026-10-31 08:21:27
3,0,3,i didn't subscribe last month and i was debite...,1,2026-10-31 08:21:27
4,0,4,i don't like it,1,2026-10-31 08:21:27
5,1,0,i can watch any movie i want,5,2026-10-31 08:20:00
6,2,0,it stole my heart,5,2026-10-31 07:52:54
7,3,0,i want to be on netflix,5,2026-10-31 07:48:10
8,4,0,download loud house for me,5,2026-10-31 07:17:34
9,5,0,"i have always loved the pivot podcast, and cor...",5,2026-10-31 07:00:46


## 5. Load SWOT Classifier

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading classifier from:", MODEL_PATH)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
classifier_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(device).eval()

model_labels = [
    classifier_model.config.id2label[i]
    for i in range(len(SWOT_LABELS))
]

assert model_labels == SWOT_LABELS, (
    f"Label mismatch: {model_labels} != {SWOT_LABELS}"
)

print("Classifier ready:", device)

@torch.inference_mode()
def predict_proba(texts, batch_size=64):
    if len(texts) == 0:
        return np.empty((0, len(SWOT_LABELS)), dtype=np.float32)

    out = []

    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]

        enc = classifier_tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        logits = classifier_model(**enc).logits

        probabilities = torch.softmax(logits.float(), dim=-1)
        out.append(probabilities.cpu().numpy())

    return np.vstack(out)

Loading classifier from: /kaggle/input/datasets/anastasiapolitidou/netflix-swot-model/swot_roberta_base_regularized/best_model_verified


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier ready: cuda


## 6. SWOT Classification

In [8]:
segments_reset = segments_df.reset_index(drop=True).copy()

p_all = predict_proba(
    segments_reset["text_clean"].astype(str).tolist()
)

pred_idx = np.argmax(p_all, axis=1)

df_swot = segments_reset.copy()
df_swot["swot_category"] = np.array(SWOT_LABELS)[pred_idx]
df_swot["swot_confidence"] = p_all[
    np.arange(len(p_all)), pred_idx
]

assert len(df_swot) == len(segments_df)
assert df_swot["swot_category"].isin(SWOT_LABELS).all()

df_swot.to_csv(
    "/kaggle/working/corpus_classified.csv",
    index=False
)

CLASSIFIED = df_swot.copy()

coverage = 1.0
macro_f1 = MACRO_F1

print("Classified segments:", len(CLASSIFIED))
print("Coverage:", f"{coverage:.1%}")
print("Test Macro F1:", f"{macro_f1:.4f}")

display(
    CLASSIFIED["swot_category"]
    .value_counts()
    .reindex(SWOT_LABELS, fill_value=0)
    .rename_axis("SWOT Category")
    .reset_index(name="Opinions")
)

monthly_classified = CLASSIFIED[
    CLASSIFIED["date"].dt.to_period("M").isin(
        [PREVIOUS_MONTH, CURRENT_MONTH]
    )
].copy()

display(
    pd.crosstab(
        monthly_classified["date"].dt.to_period("M"),
        monthly_classified["swot_category"]
    ).reindex(columns=SWOT_LABELS, fill_value=0)
)

Classified segments: 252290
Coverage: 100.0%
Test Macro F1: 0.7882


,SWOT Category,Opinions
0,Strength,66827
1,Weakness,113704
2,Opportunity,28546
3,Threat,43213


swot_category,Strength,Weakness,Opportunity,Threat
date,,,,
2026-09,1686,744,393,519
2026-10,1251,584,314,346


## 7. Sentence Embeddings

In [9]:
from sentence_transformers import SentenceTransformer




if torch.cuda.is_available():
    classifier_model.to("cpu")
gc.collect(); torch.cuda.empty_cache()

embedding_model = SentenceTransformer("all-mpnet-base-v2")


def embed_category(category):
    cat_df = CLASSIFIED[(CLASSIFIED["swot_category"] == category)
                        & CLASSIFIED["text_clean"].notna()].reset_index(drop=True)
    if len(cat_df) < 100:
        print(f"{category}: only {len(cat_df)} segments, skipped")
        return {"df": cat_df, "emb": None}
    emb = embedding_model.encode(cat_df["text_clean"].astype(str).tolist(),
                                 batch_size=256, show_progress_bar=True,
                                 normalize_embeddings=True).astype(np.float32)
    print(f"{category}: {len(cat_df):,} segments embedded")
    return {"df": cat_df, "emb": emb}


topic_data = {c: embed_category(c) for c in SWOT_LABELS}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (11 outputs left out)

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Opportunity: 28,546 segments embedded


Batches:   0%|          | 0/169 [00:00<?, ?it/s]

Threat: 43,213 segments embedded


## 8. Theme Discovery and Semantic Assignment

In [10]:
import gc, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MONTHLY_TREND_THRESHOLD = 0.01
MIN_THEME_SHARE = 0.005
ASSIGN_MIN_SIM = 0.30
MERGE_SIM = 0.67
MERGE_NAME_SIM = 0.72
NEIGHBOUR_SIM = 0.55
DIVERSITY_SIM = 0.55
REFERENCE_N = 12000
CANDIDATE_N = 30000
DESCRIBE_MIN_SIM = 0.45
BATCH = 60

THEME_ID = "Qwen/Qwen2.5-7B-Instruct"
CACHE_VERSION = "monthly_qwen7b_v6"
CACHE_ROOT = Path("/kaggle/working/netflix_theme_cache_v6")
SNAPSHOT_DIR = CACHE_ROOT / "inputs"
RESPONSE_DIR = CACHE_ROOT / "responses"
RESULT_DIR = CACHE_ROOT / "results"

for directory in (SNAPSHOT_DIR, RESPONSE_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if "topic_data" in globals():
    signatures = {}

    for category in SWOT_LABELS:
        frame = topic_data[category]["df"]
        vectors = topic_data[category]["emb"]

        if vectors is None:
            raise ValueError(f"Missing embeddings for {category}")

        cols = ["review_id", "text_clean", "date", "swot_category"]
        digest = hashlib.sha256(
            pd.util.hash_pandas_object(frame[cols], index=False).values.tobytes()
        ).hexdigest()

        signatures[category] = {
            "hash": digest,
            "rows": len(frame),
            "shape": list(vectors.shape)
        }

        vec_path = SNAPSHOT_DIR / f"{category}_emb.npy"
        df_path = SNAPSHOT_DIR / f"{category}_df.pkl"
        meta_path = SNAPSHOT_DIR / f"{category}_meta.json"

        old_meta = json.loads(meta_path.read_text()) if meta_path.exists() else None

        if old_meta != signatures[category] or not (vec_path.exists() and df_path.exists()):
            temp = SNAPSHOT_DIR / f"{category}_emb.tmp.npy"
            np.save(temp, np.asarray(vectors, dtype=np.float32))
            temp.replace(vec_path)
            frame.to_pickle(df_path)
            meta_path.write_text(json.dumps(signatures[category]))

    (SNAPSHOT_DIR / "periods.json").write_text(json.dumps({
        "current": str(CURRENT_MONTH),
        "previous": str(PREVIOUS_MONTH)
    }))

else:
    periods_file = SNAPSHOT_DIR / "periods.json"

    if not periods_file.exists():
        raise RuntimeError("Cell 7 must run once before Cell 8 to save its embeddings")

    periods = json.loads(periods_file.read_text())
    CURRENT_MONTH = pd.Period(periods["current"])
    PREVIOUS_MONTH = pd.Period(periods["previous"])
    SWOT_LABELS = ["Strength", "Weakness", "Opportunity", "Threat"]

    topic_data, signatures = {}, {}

    for category in SWOT_LABELS:
        meta = json.loads((SNAPSHOT_DIR / f"{category}_meta.json").read_text())
        frame = pd.read_pickle(SNAPSHOT_DIR / f"{category}_df.pkl")
        vectors = np.load(SNAPSHOT_DIR / f"{category}_emb.npy", mmap_mode="r")

        if len(frame) != len(vectors) or list(vectors.shape) != meta["shape"]:
            raise ValueError(f"Cached data mismatch: {category}")

        topic_data[category] = {"df": frame, "emb": vectors}
        signatures[category] = meta

run_key = hashlib.sha256(json.dumps({
    "version": CACHE_VERSION,
    "reporting_scope": "historical_unique_reviews_v2",
    "model": THEME_ID,
    "current": str(CURRENT_MONTH),
    "previous": str(PREVIOUS_MONTH),
    "signatures": signatures,
    "thresholds": [
        MIN_THEME_SHARE, ASSIGN_MIN_SIM, MERGE_SIM,
        MERGE_NAME_SIM, NEIGHBOUR_SIM, DIVERSITY_SIM, DESCRIBE_MIN_SIM
    ]
}, sort_keys=True).encode()).hexdigest()[:20]

if "classifier_model" in globals():
    classifier_model.to("cpu")

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cpu")

embedding_model.to("cpu")

for name in ("theme_model", "_theme_model", "_qwen_model", "theme_tokenizer", "_theme_tokenizer"):
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_qwen_model = None
_qwen_tokenizer = None
_qwen_device = None


def load_qwen():
    global _qwen_model, _qwen_tokenizer, _qwen_device

    if _qwen_model is not None:
        return

    if not torch.cuda.is_available():
        raise RuntimeError("GPU is required to load Qwen2.5-7B in 4-bit")

    free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
    gpu = int(np.argmax(free))

    if free[gpu] < 7 * (1024 ** 3):
        raise RuntimeError("Less than 7 GiB of free GPU memory. Restart the session to release stale allocations.")

    _qwen_device = f"cuda:{gpu}"
    _qwen_tokenizer = AutoTokenizer.from_pretrained(THEME_ID)

    if _qwen_tokenizer.pad_token is None:
        _qwen_tokenizer.pad_token = _qwen_tokenizer.eos_token

    _qwen_model = AutoModelForCausalLM.from_pretrained(
        THEME_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        ),
        device_map={"": gpu},
        low_cpu_mem_usage=True
    )
    _qwen_model.eval()


def ask_qwen(prompt, max_new_tokens=220):
    key = hashlib.sha256(json.dumps(
        [CACHE_VERSION, THEME_ID, prompt, max_new_tokens], ensure_ascii=False
    ).encode()).hexdigest()

    path = RESPONSE_DIR / f"{key}.json"

    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))["response"]
        except (ValueError, KeyError):
            path.unlink()

    load_qwen()

    messages = [
        {"role": "system", "content": "Identify specific recurring topics in customer reviews. Use only the supplied evidence."},
        {"role": "user", "content": prompt}
    ]

    template = _qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    enc = _qwen_tokenizer(
        template, truncation=True, max_length=2700, return_tensors="pt"
    ).to(_qwen_device)

    with torch.inference_mode():
        generated = _qwen_model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=_qwen_tokenizer.pad_token_id
        )

    response = _qwen_tokenizer.decode(
        generated[0, enc["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del enc, generated

    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps({"response": response}, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

    return response


def topic_trend(previous_share, current_share):
    delta = current_share - previous_share
    return "RISING" if delta >= MONTHLY_TREND_THRESHOLD else "FALLING" if delta <= -MONTHLY_TREND_THRESHOLD else "STABLE"


def representative_positions(frame, emb, theme_vector, k=6, pool=160):
    texts = frame["text_clean"].fillna("").astype(str).to_numpy()
    ids = frame["review_id"].to_numpy()
    eligible = np.flatnonzero(np.array([len(t.split()) >= 5 for t in texts]))
    if not len(eligible):
        eligible = np.arange(len(texts))
    if not len(eligible):
        return []
    vectors = np.asarray(emb[eligible], dtype=np.float32)
    centroid = vectors.mean(axis=0)
    centroid /= np.linalg.norm(centroid) + 1e-9
    relevance = 0.60 * (vectors @ centroid) + 0.40 * (vectors @ theme_vector)
    ranked = np.argsort(-relevance)[:min(pool, len(eligible))]
    picked, used_reviews, used_texts = [], set(), set()
    while len(picked) < k:
        options = []
        for i in ranked:
            j = int(eligible[i])
            if ids[j] in used_reviews or texts[j].casefold() in used_texts:
                continue
            redundancy = max((float(emb[j] @ emb[x]) for x in picked), default=0.0)
            score = 0.85 * relevance[i] - 0.15 * redundancy
            options.append((score, j))
        if not options:
            break
        selected = max(options)[1]
        picked.append(selected)
        used_reviews.add(ids[selected])
        used_texts.add(texts[selected].casefold())
    return picked

def propose_themes(reviews, category):
    examples = "\n".join(f"- {r}" for r in reviews)

    prompt = f"""These are real Netflix review segments assigned to SWOT category {category}.

{examples}

List up to 10 recurring themes shared by multiple reviews. Return one per line:
Theme Name | keyword, keyword, keyword

Rules:
- Use concrete titles of 2 to 5 words about a specific feature, issue or experience.
- Avoid vague labels such as Netflix Entertainment, Good App Reviews or Subscription Issues.
- Do not repeat the same issue under different names.
- Keep different customer issues separate, such as price increases versus unexpected charges.
- Base all themes only on the review segments. Return the lines only."""

    response = ask_qwen(prompt)
    themes = []

    for line in response.splitlines():
        line = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line.strip())
        if "|" not in line:
            continue

        name, keywords = line.split("|", 1)
        name = re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9 &'-]", " ", name)).strip()
        keywords = re.sub(r"[^A-Za-z0-9, '-]", " ", keywords).strip()

        words = name.split()
        for size in range(1, len(words) // 2 + 1):
            if len(words) % size == 0 and words == words[:size] * (len(words) // size):
                name = " ".join(words[:size])
                break

        if 2 <= len(name.split()) <= 6 and keywords:
            themes.append((name, keywords))

    return themes


def merge_proposals(themes):
    if not themes:
        return []

    names = embedding_model.encode(
        [n for n, _ in themes], normalize_embeddings=True,
        device="cpu", show_progress_bar=False
    )
    texts = embedding_model.encode(
        [f"{n}: {kw}" for n, kw in themes],
        normalize_embeddings=True, device="cpu", show_progress_bar=False
    )

    groups = []

    for i in range(len(themes)):
        group = next((
            g for g in groups
            if float(names[i] @ names[g[0]]) >= MERGE_NAME_SIM
            and float(texts[i] @ texts[g[0]]) >= MERGE_SIM
        ), None)

        if group is None:
            groups.append([i])
        else:
            group.append(i)

    return [
        themes[max(g, key=lambda i: (len(themes[i][0].split()), len(themes[i][1])))]
        for g in groups
    ]


def popular_sample(cat_df, emb, k):
    idx = np.flatnonzero(
        cat_df["text_clean"].astype(str).str.split().str.len().ge(6).to_numpy()
    )

    if len(idx) <= k:
        return cat_df["text_clean"].astype(str).iloc[idx].tolist()

    ref = idx[np.linspace(0, len(idx) - 1, min(REFERENCE_N, len(idx))).astype(int)]
    cand = idx[np.linspace(0, len(idx) - 1, min(CANDIDATE_N, len(idx))).astype(int)]
    density = np.empty(len(cand), dtype=np.int32)
    ref_vectors = np.asarray(emb[ref], dtype=np.float32)

    for start in range(0, len(cand), 500):
        density[start:start + 500] = (
            emb[cand[start:start + 500]] @ ref_vectors.T >= NEIGHBOUR_SIM
        ).sum(axis=1)

    texts = cat_df["text_clean"].astype(str).to_numpy()
    selected, vectors = [], []

    for j in np.argsort(-density):
        v = emb[cand[j]]
        if vectors and max(float(v @ other) for other in vectors) >= DIVERSITY_SIM:
            continue

        selected.append(texts[cand[j]])
        vectors.append(v)

        if len(selected) >= k:
            break

    return selected


def group_related_themes(candidates, centroids, name_emb, full_emb):
    order = sorted(
        range(len(candidates)),
        key=lambda i: candidates[i]["historical_mentions"],
        reverse=True
    )
    groups = []

    for i in order:
        for group in groups:
            match = True

            for j in group:
                name_sim = float(name_emb[i] @ name_emb[j])
                full_sim = float(full_emb[i] @ full_emb[j])
                data_sim = float(centroids[i] @ centroids[j])

                similar = (
                    (name_sim >= 0.60 and full_sim >= 0.67 and data_sim >= 0.76) or
                    (name_sim >= 0.72 and data_sim >= 0.72) or
                    (name_sim >= 0.50 and full_sim >= 0.75 and data_sim >= 0.86)
                )

                if not similar:
                    match = False
                    break

            if match:
                group.append(i)
                break
        else:
            groups.append([i])

    return groups


def build_themes(category):
    cat_df, emb = topic_data[category]["df"], topic_data[category]["emb"]

    if emb is None or len(cat_df) < 100:
        return pd.DataFrame(), {}, []

    cat_df = cat_df.reset_index(drop=True)
    period = cat_df["date"].dt.to_period("M")
    old_mask = (period == PREVIOUS_MONTH).to_numpy()
    new_mask = (period == CURRENT_MONTH).to_numpy()

    sample_file = RESULT_DIR / f"{category}_{run_key}_sample.json"

    if sample_file.exists():
        examples = json.loads(sample_file.read_text(encoding="utf-8"))
    else:
        historical = popular_sample(cat_df, emb, k=300)
        previous = popular_sample(
            cat_df.loc[old_mask].reset_index(drop=True), emb[old_mask], k=130
        )
        current = popular_sample(
            cat_df.loc[new_mask].reset_index(drop=True), emb[new_mask], k=130
        )

        examples = list(dict.fromkeys(historical + previous + current))
        tmp = sample_file.with_suffix(".tmp")
        tmp.write_text(json.dumps(examples, ensure_ascii=False), encoding="utf-8")
        tmp.replace(sample_file)

    proposals = []

    for start in range(0, len(examples), BATCH):
        proposals.extend(propose_themes(examples[start:start + BATCH], category))

    themes = merge_proposals(proposals)

    if not themes:
        raise RuntimeError(f"Qwen returned no valid themes for {category}")

    print(f"{category}: {len(proposals)} proposals, {len(themes)} distinct")

    theme_emb = embedding_model.encode(
        [f"{name}: {keywords}" for name, keywords in themes],
        normalize_embeddings=True, device="cpu", show_progress_bar=False
    ).astype(np.float32)

    assignments = np.full(len(cat_df), -1, dtype=np.int32)

    for start in range(0, len(cat_df), 5000):
        similarities = emb[start:start + 5000] @ theme_emb.T
        best = similarities.argmax(axis=1)
        score = similarities[np.arange(len(best)), best]
        assignments[start:start + len(best)] = np.where(score >= ASSIGN_MIN_SIM, best, -1)

    previous_total = cat_df.loc[old_mask, "review_id"].nunique()
    current_total = cat_df.loc[new_mask, "review_id"].nunique()
    historical_total = cat_df["review_id"].nunique()

    if previous_total == 0 or current_total == 0:
        raise RuntimeError(f"No comparable monthly reviews for {category}")

    min_recent = max(10, int(MIN_THEME_SHARE * min(previous_total, current_total)))
    min_historical = max(10, int(MIN_THEME_SHARE * historical_total))
    candidates, candidate_centroids = [], []

    for theme_id, (name, keywords) in enumerate(themes):
        mask = assignments == theme_id
        if not mask.any():
            continue
        unique_count = cat_df.loc[mask, "review_id"].nunique()
        centroid = emb[mask].mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
        candidates.append({
            "id": theme_id,
            "name": name,
            "keywords": keywords,
            "historical_mentions": int(unique_count)
        })
        candidate_centroids.append(centroid)

    if not candidates:
        raise ValueError(f"No assigned candidate themes for {category}")

    name_vectors = embedding_model.encode(
        [x["name"] for x in candidates], normalize_embeddings=True, device="cpu"
    )
    full_vectors = embedding_model.encode(
        [f'{x["name"]}: {x["keywords"]}' for x in candidates],
        normalize_embeddings=True, device="cpu"
    )

    groups = group_related_themes(
        candidates, candidate_centroids, name_vectors, full_vectors
    )
    rows, centroids, merge_log = [], {}, []
    topic_index = np.full(len(cat_df), -1, dtype=np.int32)
    assignment_scores = np.zeros(len(cat_df), dtype=np.float32)
    for start in range(0, len(cat_df), 5000):
        sims = emb[start:start + 5000] @ theme_emb.T
        assignment_scores[start:start + len(sims)] = np.max(sims, axis=1)

    for group in groups:
        member_ids = [candidates[i]["id"] for i in group]
        mask = np.isin(assignments, member_ids)

        relevant = cat_df.loc[mask]
        relevant_month = relevant["date"].dt.to_period("M")
        old = relevant.loc[relevant_month == PREVIOUS_MONTH]
        new = relevant.loc[relevant_month == CURRENT_MONTH]

        previous_count = int(old["review_id"].nunique())
        current_count = int(new["review_id"].nunique())

        historical_count = int(relevant["review_id"].nunique())
        if historical_count < min_historical and max(current_count, previous_count) < min_recent:
            continue

        old_share = previous_count / previous_total
        new_share = current_count / current_total

        evidence_df = relevant.reset_index(drop=True)
        evidence_emb = emb[mask]
        representative = max(group, key=lambda i: candidates[i]["historical_mentions"])
        reference_vector = theme_emb[candidates[representative]["id"]]
        chosen = representative_positions(evidence_df, evidence_emb, reference_vector, k=8)
        quote_positions = chosen[:6]
        evidence_texts = evidence_df["text_clean"].astype(str).tolist()
        centroid = evidence_emb.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)

        topic_name = candidates[representative]["name"]
        if topic_name in centroids:
            topic_name += f" ({len(centroids) + 1})"

        centroids[topic_name] = centroid

        if len(group) > 1:
            merge_log.append((topic_name, [candidates[i]["name"] for i in group]))

        topic_index[mask] = len(rows)
        row = {
            "topic_name": topic_name,
            "central_texts": " || ".join(evidence_texts[i] for i in chosen),
            "topic_phrase": candidates[representative]["keywords"],
            "mention_count": current_count,
            "previous_mention_count": previous_count,
            "historical_mention_count": historical_count,
            "segment_count": int(len(new)),
            "historical_segment_count": int(len(relevant)),
            "current_share": round(float(new_share), 5),
            "previous_share": round(float(old_share), 5),
            "mom_change_pp": round(float((new_share - old_share) * 100), 2),
            "coherence": round(float((evidence_emb @ centroid).mean()), 3),
            "avg_rating": round(float(evidence_df.drop_duplicates("review_id")["rating"].mean()), 2),
            "current_avg_rating": round(float(new.drop_duplicates("review_id")["rating"].mean()), 2) if len(new) else None,
            "avg_confidence": round(float(evidence_df["swot_confidence"].mean()), 3),
            "trend": topic_trend(old_share, new_share)
        }

        for qi in range(6):
            pos = quote_positions[qi] if qi < len(quote_positions) else None
            row[f"quote_{qi + 1}"] = evidence_texts[pos] if pos is not None else ""
            row[f"quote_review_id_{qi + 1}"] = int(evidence_df.iloc[pos]["review_id"]) if pos is not None else None

        row["_topic_row"] = len(rows)
        rows.append(row)

    table = pd.DataFrame(rows)

    member_mask = topic_index >= 0
    member_frame = cat_df.loc[member_mask, [
        "review_id", "segment_id", "date", "text_clean", "rating", "swot_confidence"
    ]].copy()
    member_frame["_topic_row"] = topic_index[member_mask]
    member_frame["similarity"] = np.round(assignment_scores[member_mask], 4)

    if not table.empty:
        table = table.sort_values("historical_mention_count", ascending=False).reset_index(drop=True)
        assert table["current_share"].between(0, 1).all()
        assert table["previous_share"].between(0, 1).all()
        assert table["historical_mention_count"].le(historical_total).all()
        assert table["mention_count"].le(current_total).all()
        assert table["previous_mention_count"].le(previous_total).all()

    print(f"{category}: {(assignments != -1).mean():.1%} semantically matched, {len(table)} retained themes")
    return table, centroids, merge_log, member_frame


def label_theme(row, centroid):
    texts = [
        text.strip() for text in row["central_texts"].split(" || ") if text.strip()
    ][:8]

    evidence = "\n".join(f"- {text}" for text in texts)

    prompt = f"""These customer review segments are assigned to one recurring topic:

{evidence}

Return exactly two lines:
NAME: a specific title of 2 to 5 words about this recurring experience or issue
DESC: one sentence of 12 to 25 words summarising what these users say

Avoid vague titles and unsupported claims. Return only the two lines."""

    for attempt in range(2):
        question = prompt if attempt == 0 else (
            prompt + "\nUse exactly NAME: and DESC: on separate lines."
        )

        raw = ask_qwen(question, max_new_tokens=110)

        name = next((
            x.strip()[5:].strip().strip('"')
            for x in raw.splitlines()
            if x.strip().upper().startswith("NAME:")
        ), "")

        desc = next((
            x.strip()[5:].strip().strip('"')
            for x in raw.splitlines()
            if x.strip().upper().startswith("DESC:")
        ), "")

        name = re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9 &'-]", " ", name)).strip()

        if not (2 <= len(name.split()) <= 5 and 8 <= len(desc.split()) <= 35):
            continue

        if any(ord(ch) > 591 for ch in name + desc):
            continue

        pair = embedding_model.encode(
            [f"{name}: {desc}", name, desc],
            normalize_embeddings=True, device="cpu"
        )

        if float(pair[1] @ pair[2]) >= 0.40 and float(pair[0] @ centroid) >= DESCRIBE_MIN_SIM:
            title = " ".join(
                word.capitalize() if word.islower() else word for word in name.split()
            )
            return title, desc

    return row["topic_name"], f"Reviews mention {row['topic_name'].lower()}."


topic_tables, theme_centroids, theme_merges, theme_assignments = {}, {}, {}, {}

for category in SWOT_LABELS:
    saved = RESULT_DIR / f"{category}_{run_key}.pkl"

    if saved.exists():
        try:
            result = pd.read_pickle(saved)
        except Exception:
            saved.unlink()
            result = None
    else:
        result = None

    if result is not None:
        table, centroids, merge_log, members = (
            result["table"], result["centroids"], result["merges"], result["assignments"]
        )
        print(f"{category}: restored completed results")

    else:
        table, centroids, merge_log, members = build_themes(category)

        if not table.empty:
            labelled = [
                label_theme(row, centroids[row["topic_name"]])
                for _, row in table.iterrows()
            ]

            used, names = set(), []

            for original, (candidate, _) in zip(table["topic_name"], labelled):
                chosen = candidate if candidate not in used else original
                if chosen in used:
                    chosen = f"{original} ({len(used) + 1})"

                used.add(chosen)
                names.append(chosen)

            original_names = table["topic_name"].tolist()
            centroids = {
                new: centroids[old]
                for old, new in zip(original_names, names)
            }

            table["topic_name"] = names
            table["business_insight"] = [desc for _, desc in labelled]
            mapping = dict(zip(table["_topic_row"], table["topic_name"]))
            members["topic_name"] = members["_topic_row"].map(mapping)
            members = members.drop(columns=["_topic_row"])
            table = table.drop(columns=["_topic_row"])

        temp = saved.with_suffix(".tmp")
        pd.to_pickle({
            "table": table,
            "centroids": centroids,
            "merges": merge_log,
            "assignments": members
        }, temp)
        temp.replace(saved)

    topic_tables[category] = table
    theme_centroids[category] = centroids
    theme_merges[category] = merge_log
    theme_assignments[category] = members
    print(f"{category}: {len(table)} themes")

    if not table.empty:
        display(
            table[
                ["topic_name", "historical_mention_count", "previous_mention_count",
                 "mention_count", "mom_change_pp", "trend"]
            ].head(10)
        )

    for name, members in merge_log[:8]:
        print("Merged:", " + ".join(members), "->", name)

if _qwen_model is not None:
    del _qwen_model
    _qwen_model = None
    gc.collect()
    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

,topic_name,historical_mention_count,previous_mention_count,mention_count,mom_change_pp,trend
0,Limited Content Availability,2392,28,13,-3.30,FALLING
1,Missing Shows and Movies,2297,34,22,-1.77,FALLING
2,More Movie Options,2022,32,34,3.10,RISING
3,Tamil Dubbing Requests,1080,21,17,0.13,STABLE
4,Subtitle Language Options,1065,10,3,-1.77,FALLING
5,Missing Seasons,1013,31,27,0.87,STABLE
6,Regional Content Availability,1011,14,10,-0.39,STABLE
7,Anime Content Request,872,13,3,-2.62,FALLING
8,Watch Later Improvements,716,3,7,1.66,RISING
9,Streaming Quality Control,599,2,5,1.22,RISING


Merged: Subtitles and Language Options + Language Options -> Subtitles and Language Options
Merged: Seasons and Episodes + Seasons and Continuation -> Seasons and Episodes
Merged: More Content + More Animation Content -> More Content
Merged: Anime Selection + Anime Specific Requests -> Anime Selection
Merged: Classic and Vintage Movies + Classic Movies -> Classic and Vintage Movies
Merged: Download Option + Download and Storage Options -> Download Option
Merged: More Horror Content + More Horror and Comedy Movies -> More Horror Content
Merged: Improved User Interface + User Interface Improvements -> Improved User Interface
Threat: 96 proposals, 67 distinct
Threat: 86.7% semantically matched, 37 retained themes
Threat: 37 themes


,topic_name,historical_mention_count,previous_mention_count,mention_count,mom_change_pp,trend
0,Auto-Renewing Subscriptions,5213,48,37,1.78,RISING
1,Subscription Cancellation Issues,3187,24,23,2.72,RISING
2,Content Quality Decline,2645,33,16,-2.61,FALLING
3,High Content Costs,2562,34,24,0.36,STABLE
4,High Pricing Complaints,2439,27,16,-0.95,STABLE
5,Household Subscription Issues,1558,5,13,3.90,RISING
6,Expensive Subscription,1439,13,8,-0.34,STABLE
7,App Quality Deterioration,1401,10,10,1.30,RISING
8,Content Quality Issues,1263,11,7,-0.19,STABLE
9,Show Cancellations And Removals,1132,11,4,-1.41,FALLING


Merged: Subscription Issues + Subscription Changes -> Subscription Issues
Merged: App Performance + App Performance -> App Performance
Merged: Subscription Restrictions + Subscription Sharing -> Subscription Restrictions
Merged: Content Quality and Availability + Content Availability -> Content Quality and Availability
Merged: Price Increases + High Prices -> Price Increases
Merged: Service Disappointment + Poor Customer Service -> Service Disappointment
Merged: Account Management Difficulties + Account Management + Account Management -> Account Management Difficulties
Merged: Inconvenient Renewals + Auto-Renewal Issues -> Inconvenient Renewals


In [11]:
theme_overview = pd.concat([
    topic_tables[c].assign(swot=c) for c in SWOT_LABELS if not topic_tables[c].empty
], ignore_index=True)
display(theme_overview[["swot", "topic_name", "historical_mention_count", "mention_count", "trend"]].head(30))

,swot,topic_name,historical_mention_count,mention_count,trend
0,Strength,User-Friendly Interface,18414,166,FALLING
1,Strength,Movie And TV Library,3524,73,STABLE
2,Strength,Streaming Quality,3242,30,FALLING
3,Strength,Longtime Satisfaction,2481,23,STABLE
4,Strength,Movie & Show Variety,2460,39,FALLING
5,Strength,Netflix Movie Nights,2133,38,STABLE
6,Strength,Best Entertainment App,2020,23,STABLE
7,Strength,Mobile App Usability,1440,8,STABLE
8,Strength,Binge Watching Hub,1090,14,STABLE
9,Strength,Stranger Things Fans,1052,12,STABLE


## 8.1. Theme Assignment Validation

In [12]:
import pandas as pd
from pathlib import Path

validation_frames = []
for category in SWOT_LABELS:
    table = topic_tables[category]
    assigned = theme_assignments[category]
    assert not table.empty and not assigned.empty, f"Missing themes or assignments: {category}"
    assert assigned["topic_name"].notna().all()
    assert assigned["topic_name"].isin(table["topic_name"]).all()
    period = assigned["date"].dt.to_period("M")
    for row in table.itertuples(index=False):
        matched = assigned.loc[assigned["topic_name"] == row.topic_name]
        months = matched["date"].dt.to_period("M")
        historical = int(matched["review_id"].nunique())
        previous = int(matched.loc[months == PREVIOUS_MONTH, "review_id"].nunique())
        current = int(matched.loc[months == CURRENT_MONTH, "review_id"].nunique())
        assert historical == int(row.historical_mention_count), row.topic_name
        assert previous == int(row.previous_mention_count), row.topic_name
        assert current == int(row.mention_count), row.topic_name
        validation_frames.append({
            "swot": category, "topic_name": row.topic_name,
            "historical_reviews": historical, "previous_reviews": previous,
            "current_reviews": current,
            "mean_assignment_similarity": round(float(matched["similarity"].mean()), 3),
            "minimum_assignment_similarity": round(float(matched["similarity"].min()), 3)
        })

validation_df = pd.DataFrame(validation_frames)
validation_df.to_csv("/kaggle/working/theme_assignment_validation.csv", index=False)
assert len(validation_df) == sum(len(t) for t in topic_tables.values())
print("Historical and monthly theme counts verified:", len(validation_df))
display(validation_df.head(20))

Historical and monthly theme counts verified: 168


,swot,topic_name,historical_reviews,previous_reviews,current_reviews,mean_assignment_similarity,minimum_assignment_similarity
0,Strength,User-Friendly Interface,18414,256,166,0.531,0.300
1,Strength,Movie And TV Library,3524,105,73,0.529,0.300
2,Strength,Streaming Quality,3242,67,30,0.500,0.300
3,Strength,Longtime Satisfaction,2481,35,23,0.467,0.301
4,Strength,Movie & Show Variety,2460,84,39,0.549,0.301
5,Strength,Netflix Movie Nights,2133,45,38,0.506,0.300
6,Strength,Best Entertainment App,2020,25,23,0.465,0.302
7,Strength,Mobile App Usability,1440,14,8,0.400,0.300
8,Strength,Binge Watching Hub,1090,20,14,0.495,0.301
9,Strength,Stranger Things Fans,1052,22,12,0.484,0.301


## 9. Historical SWOT Matrix

In [13]:
import numpy as np
import pandas as pd

HISTORICAL_CLASSIFIED = CLASSIFIED.loc[CLASSIFIED["date"].lt(REPORTING_END)].copy()
CURRENT_CLASSIFIED = HISTORICAL_CLASSIFIED.loc[
    HISTORICAL_CLASSIFIED["date"].dt.to_period("M") == CURRENT_MONTH
].copy()
QUAD_TOTALS = {
    c: int(HISTORICAL_CLASSIFIED.loc[
        HISTORICAL_CLASSIFIED["swot_category"] == c, "review_id"
    ].nunique()) for c in SWOT_LABELS
}
MONTHLY_QUAD_TOTALS = {
    c: int(CURRENT_CLASSIFIED.loc[
        CURRENT_CLASSIFIED["swot_category"] == c, "review_id"
    ].nunique()) for c in SWOT_LABELS
}

final_rows, selection_thresholds = [], {}
for category in SWOT_LABELS:
    table = topic_tables[category]
    if table.empty:
        raise RuntimeError(f"No discovered themes for {category}")
    frequencies = table["historical_mention_count"].astype(float)
    threshold = float(frequencies.quantile(0.75))
    selection_thresholds[category] = threshold
    chosen = table.loc[frequencies.ge(threshold)].copy()
    chosen = chosen.sort_values(["historical_mention_count", "topic_name"], ascending=[False, True])
    chosen["swot"] = category
    chosen["current_mention_count"] = chosen["mention_count"]
    chosen["mention_count"] = chosen["historical_mention_count"]
    chosen["share"] = (chosen["mention_count"] / QUAD_TOTALS[category] * 100).round(1)
    q_lo, q_hi = chosen["share"].quantile([1 / 3, 2 / 3])
    chosen["priority"] = np.select(
        [chosen["share"] >= q_hi, chosen["share"] >= q_lo],
        ["HIGH", "MEDIUM"], default="LOW"
    )
    assert chosen["share"].between(0, 100).all()
    final_rows.append(chosen)

final_swot = pd.concat(final_rows, ignore_index=True)
final_swot = final_swot.sort_values(
    ["swot", "mention_count"], ascending=[True, False]
).reset_index(drop=True)
final_swot.to_csv("/kaggle/working/final_swot_matrix.csv", index=False)
print("Historical SWOT period: Jan 2020 to", CURRENT_MONTH)
print("Discovered themes:", sum(len(t) for t in topic_tables.values()))
print("Selected themes:", len(final_swot))
display(final_swot[[
    "swot", "topic_name", "mention_count", "share", "current_mention_count",
    "previous_mention_count", "trend", "priority"
]])

Historical SWOT period: Jan 2020 to 2026-10
Discovered themes: 168
Selected themes: 45


,swot,topic_name,mention_count,share,current_mention_count,previous_mention_count,trend,priority
0,Opportunity,Limited Content Availability,2392,10.2,13,28,FALLING,HIGH
1,Opportunity,Missing Shows and Movies,2297,9.8,22,34,FALLING,HIGH
2,Opportunity,More Movie Options,2022,8.6,34,32,RISING,HIGH
3,Opportunity,Tamil Dubbing Requests,1080,4.6,17,21,STABLE,HIGH
4,Opportunity,Subtitle Language Options,1065,4.5,3,10,FALLING,MEDIUM
5,Opportunity,Missing Seasons,1013,4.3,27,31,STABLE,MEDIUM
6,Opportunity,Regional Content Availability,1011,4.3,10,14,STABLE,MEDIUM
7,Opportunity,Anime Content Request,872,3.7,3,13,FALLING,LOW
8,Opportunity,Watch Later Improvements,716,3.0,7,3,RISING,LOW
9,Opportunity,Streaming Quality Control,599,2.5,5,2,RISING,LOW


## 10. Representative Review Evidence

In [14]:
from pathlib import Path
import pandas as pd

report_lines, evidence_rows = [], []
for category in SWOT_LABELS:
    assigned = theme_assignments[category]
    table = topic_tables[category].set_index("topic_name")
    for row in final_swot.loc[final_swot["swot"] == category].itertuples(index=False):
        source = table.loc[row.topic_name]
        members = assigned.loc[assigned["topic_name"] == row.topic_name]
        assert members["review_id"].nunique() == row.mention_count
        report_lines.append(f"{category} / {row.topic_name} / {row.mention_count} historical reviews")
        for i in range(1, 7):
            quote = str(source.get(f"quote_{i}", "")).strip()
            rid = source.get(f"quote_review_id_{i}")
            if not quote or pd.isna(rid):
                continue
            matched = members.loc[
                (members["review_id"] == int(rid)) & (members["text_clean"] == quote)
            ]
            assert not matched.empty, f"Unmatched evidence: {row.topic_name} / {i}"
            item = matched.iloc[0]
            evidence_rows.append({
                "swot": category, "topic_name": row.topic_name, "rank": i,
                "review_id": int(item["review_id"]), "date": str(item["date"].date()),
                "rating": int(item["rating"]), "text_clean": quote,
                "assignment_similarity": float(item["similarity"])
            })
            report_lines.append(f"  {i}. [{item['date'].date()}] {quote}")
        report_lines.append("")

representative_evidence = pd.DataFrame(evidence_rows)
assert representative_evidence.groupby(["swot", "topic_name"]).size().min() >= 1
assert not representative_evidence.duplicated(["swot", "topic_name", "review_id"]).any()
representative_evidence.to_csv("/kaggle/working/theme_evidence.csv", index=False)
Path("/kaggle/working/theme_evidence.txt").write_text("\n".join(report_lines), encoding="utf-8")
print("Representative review excerpts verified:", len(representative_evidence))
display(representative_evidence.head(20))

Representative review excerpts verified: 270


,swot,topic_name,rank,review_id,date,rating,text_clean,assignment_similarity
0,Strength,User-Friendly Interface,1,55585,2020-12-14,5,"great app for movies, series, i love it and on...",0.7921
1,Strength,User-Friendly Interface,2,56122,2020-12-22,5,"i love this app, netflix is the best in movies...",0.7469
2,Strength,User-Friendly Interface,3,9764,2026-02-17,5,"the app is smooth, user-friendly, and has a hu...",0.7757
3,Strength,User-Friendly Interface,4,84277,2022-03-14,5,nice app easy to access and lot of fun movies ...,0.7741
4,Strength,User-Friendly Interface,5,109124,2021-10-08,5,i like this app alot it's easy to use and alwa...,0.7631
5,Strength,User-Friendly Interface,6,77135,2021-03-04,4,over all it is a great app beacuse there are v...,0.7913
6,Strength,Movie And TV Library,1,68501,2020-05-18,5,great library of movies and series,0.9226
7,Strength,Movie And TV Library,2,48460,2020-07-01,5,netflix has an awesome collection of shows and...,0.6850
8,Strength,Movie And TV Library,3,65221,2022-12-30,5,great library of shows and movies,0.8689
9,Strength,Movie And TV Library,4,92846,2020-04-01,5,love netflix great films an box sets!,0.6984


## 11. Export Dashboard Data

In [15]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def as_python(value):
    if isinstance(value, dict):
        return {str(k): as_python(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [as_python(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

raw_history = df_raw.loc[df_raw["date"].lt(REPORTING_END)].copy()
classified_history = HISTORICAL_CLASSIFIED.copy()
raw_history["month"] = raw_history["date"].dt.to_period("M")
raw_history["quarter"] = raw_history["date"].dt.to_period("Q")
classified_history["month"] = classified_history["date"].dt.to_period("M")
classified_history["quarter"] = classified_history["date"].dt.to_period("Q")
assert not raw_history.empty and not classified_history.empty
assert raw_history["rating"].notna().all()
assert classified_history["swot_category"].isin(SWOT_LABELS).all()

months = pd.period_range(raw_history["month"].min(), CURRENT_MONTH, freq="M")
history = []
for month in months[-6:]:
    upto = raw_history.loc[raw_history["month"] <= month]
    history.append({
        "total_reviews": int(len(upto)),
        "avg_rating": round(float(upto["rating"].mean()), 2),
        "positive_share_pct": round(float(upto["rating"].ge(4).mean() * 100), 1)
    })

ratings = raw_history["rating"]
current_month_ratings = raw_history.loc[
    raw_history["month"].eq(CURRENT_MONTH), "rating"
]
assert len(current_month_ratings) > 0, "No ratings for the reporting month"
monthly_average_rating = float(current_month_ratings.mean())
historical_average_rating = float(ratings.mean())
meta = {
    "total_reviews": int(len(raw_history)),
    "total_opinions": int(len(classified_history)),
    "avg_rating": round(historical_average_rating, 2),
    "monthly_avg_rating": round(monthly_average_rating, 2),
    "monthly_rating_review_count": int(len(current_month_ratings)),
    "positive_share_pct": round(float(ratings.ge(4).mean() * 100), 1),
    "macro_f1": round(float(MACRO_F1), 2)
}
assert history[-1]["total_reviews"] == meta["total_reviews"]
assert history[-1]["avg_rating"] == meta["avg_rating"]

historical_opinions = classified_history["swot_category"].value_counts().reindex(SWOT_LABELS, fill_value=0)
monthly_opinions = CURRENT_CLASSIFIED["swot_category"].value_counts().reindex(SWOT_LABELS, fill_value=0)
monthly_stats = {}
for month in [PREVIOUS_MONTH, CURRENT_MONTH]:
    monthly_stats[str(month)] = {
        "raw_reviews": int(raw_history["month"].eq(month).sum()),
        "cleaned_reviews": int(df["review_month"].eq(month).sum()),
        "opinion_segments": int(classified_history["month"].eq(month).sum())
    }

previous_raw = raw_history.loc[raw_history["month"].le(PREVIOUS_MONTH)]
previous_classified = classified_history.loc[classified_history["month"].le(PREVIOUS_MONTH)]
assert len(previous_raw) and len(previous_classified)
previous_rating = previous_raw["rating"]
monthly_reviews_previous = monthly_stats[str(PREVIOUS_MONTH)]["raw_reviews"]
monthly_opinions_previous = monthly_stats[str(PREVIOUS_MONTH)]["opinion_segments"]
assert monthly_reviews_previous and monthly_opinions_previous
kpi_changes = {
    "review_growth_pct": round(100 * (len(raw_history) / len(previous_raw) - 1), 2),
    "opinion_growth_pct": round(100 * (len(classified_history) / len(previous_classified) - 1), 2),
    "avg_rating_delta": round(float(ratings.mean() - previous_rating.mean()), 2),
    "monthly_rating_vs_historical": round(monthly_average_rating - historical_average_rating, 2),
    "positive_share_delta_pp": round(float(100 * (ratings.ge(4).mean() - previous_rating.ge(4).mean())), 2),
    "monthly_reviews_delta_pct": round(100 * (monthly_stats[str(CURRENT_MONTH)]["raw_reviews"] / monthly_reviews_previous - 1), 2),
    "monthly_opinions_delta_pct": round(100 * (monthly_stats[str(CURRENT_MONTH)]["opinion_segments"] / monthly_opinions_previous - 1), 2)
}

prefix = {"Strength": "s", "Weakness": "w", "Opportunity": "o", "Threat": "t"}
counters = dict.fromkeys(SWOT_LABELS, 0)
topics = []
for row in final_swot.itertuples(index=False):
    category, name = row.swot, row.topic_name
    original = topic_tables[category].set_index("topic_name").loc[name]
    assigned = theme_assignments[category]
    matches = assigned.loc[assigned["topic_name"] == name]
    quotes = []
    for i in range(1, 7):
        quote = str(original.get(f"quote_{i}", "")).strip()
        rid = original.get(f"quote_review_id_{i}")
        if not quote or pd.isna(rid):
            continue
        found = matches.loc[matches["review_id"].eq(int(rid)) & matches["text_clean"].eq(quote)]
        assert not found.empty, f"Missing assigned quote: {category} / {name}"
        quote_row = found.iloc[0]
        quotes.append([
            int(quote_row["rating"]), quote,
            pd.Timestamp(quote_row["date"]).strftime("%b %Y")
        ])
    assert quotes, f"Missing excerpts for {category} / {name}"
    counters[category] += 1
    topics.append({
        "id": f"{prefix[category]}{counters[category]}",
        "quad": category,
        "name": str(name),
        "phrase": str(row.topic_phrase),
        "mentions": int(row.mention_count),
        "avgRating": round(float(row.avg_rating), 1),
        "trend": str(row.trend),
        "prio": str(row.priority),
        "insight": str(row.business_insight),
        "quotes": quotes
    })

ct = pd.crosstab(classified_history["rating"], classified_history["swot_category"])
ct = ct.reindex(index=[1, 2, 3, 4, 5], columns=SWOT_LABELS, fill_value=0)
ct_pct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
heatmap = {c: [int(round(ct_pct.loc[r, c])) for r in range(1, 6)] for c in SWOT_LABELS}
rating_counts = raw_history["rating"].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
rating_counts = [int(rating_counts.loc[r]) for r in range(1, 6)]

last_complete_quarter = CURRENT_MONTH.asfreq("Q")
if CURRENT_MONTH.month % 3:
    last_complete_quarter -= 1
trend_quarters = pd.period_range("2021Q1", last_complete_quarter, freq="Q")
trend_count = pd.crosstab(classified_history["quarter"], classified_history["swot_category"])
trend_count = trend_count.reindex(index=trend_quarters, columns=SWOT_LABELS, fill_value=0)
trend_pct = trend_count.div(trend_count.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
trend = {c: [round(float(trend_pct.loc[q, c]), 1) for q in trend_quarters] for c in SWOT_LABELS}

volume_quarters = pd.period_range("2020Q1", last_complete_quarter, freq="Q")
review_volume = raw_history["quarter"].value_counts().reindex(volume_quarters, fill_value=0)
volume_counts = [int(review_volume.loc[q]) for q in volume_quarters]
volume = {
    "counts": [round(v / 1000, 1) for v in volume_counts],
    "reviewCounts": volume_counts,
    "quarters": [str(q) for q in volume_quarters],
    "unit": "thousands_of_reviews"
}

momentum_rows = pd.concat([
    topic_tables[c].assign(swot=c) for c in SWOT_LABELS if not topic_tables[c].empty
], ignore_index=True)
def momentum_items(table):
    return [{
        "swot": str(r.swot), "topic_name": str(r.topic_name),
        "mentions": int(r.mention_count), "change_pp": float(r.mom_change_pp),
        "trend": str(r.trend), "insight": str(r.business_insight)
    } for r in table.itertuples(index=False)]
rising = momentum_rows.loc[momentum_rows["trend"] == "RISING"].sort_values("mom_change_pp", ascending=False)
falling = momentum_rows.loc[momentum_rows["trend"] == "FALLING"].sort_values("mom_change_pp")
summary = {}
for category in SWOT_LABELS:
    leading = final_swot.loc[final_swot["swot"] == category].sort_values("mention_count", ascending=False).iloc[0]
    summary[category] = {
        "topic_name": str(leading["topic_name"]),
        "mentions": int(leading["mention_count"]),
        "trend": str(leading["trend"]),
        "insight": str(leading["business_insight"]),
        "avg_rating": float(leading["avg_rating"])
    }

data_json = as_python({
    "generated_at": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC"),
    "period_label": f"Jan 2020 – {CURRENT_MONTH.to_timestamp().strftime('%b %Y')}",
    "reportingMonth": str(CURRENT_MONTH),
    "previousMonth": str(PREVIOUS_MONTH),
    "lastCompleteQuarter": str(last_complete_quarter),
    "swotScope": "historical",
    "themeSelection": {"method": "upper_quartile_of_historical_review_frequency", "thresholds": selection_thresholds},
    "meta": meta,
    "history": history,
    "quadTotals": QUAD_TOTALS,
    "monthlyReviewTotals": MONTHLY_QUAD_TOTALS,
    "opinionTotals": {c: int(historical_opinions[c]) for c in SWOT_LABELS},
    "monthlyOpinionTotals": {c: int(monthly_opinions[c]) for c in SWOT_LABELS},
    "monthlyStats": monthly_stats,
    "kpiChanges": kpi_changes,
    "themeCount": int(sum(len(topic_tables[c]) for c in SWOT_LABELS)),
    "displayedThemeCount": len(topics),
    "topics": topics,
    "summary": summary,
    "momentum": {"rising": momentum_items(rising), "falling": momentum_items(falling)},
    "heatmap": heatmap,
    "ratingCounts": rating_counts,
    "trend": trend,
    "trendQuarters": [str(q) for q in trend_quarters],
    "volume": volume
})

output = Path("/kaggle/working/data.json")
with output.open("w", encoding="utf-8") as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2, allow_nan=False)

print("Saved:", output)
print("Historical SWOT:", data_json["period_label"])
print("Discovered themes:", data_json["themeCount"])
print("Displayed themes:", data_json["displayedThemeCount"])
display(pd.DataFrame([meta]))
display(final_swot[["swot", "topic_name", "mention_count", "share", "trend"]])

Saved: /kaggle/working/data.json
Historical SWOT: Jan 2020 – Oct 2026
Discovered themes: 168
Displayed themes: 45


,total_reviews,total_opinions,avg_rating,monthly_avg_rating,monthly_rating_review_count,positive_share_pct,macro_f1
0,142474,252290,2.93,4.26,3022,44.5,0.79


,swot,topic_name,mention_count,share,trend
0,Opportunity,Limited Content Availability,2392,10.2,FALLING
1,Opportunity,Missing Shows and Movies,2297,9.8,FALLING
2,Opportunity,More Movie Options,2022,8.6,RISING
3,Opportunity,Tamil Dubbing Requests,1080,4.6,STABLE
4,Opportunity,Subtitle Language Options,1065,4.5,FALLING
5,Opportunity,Missing Seasons,1013,4.3,STABLE
6,Opportunity,Regional Content Availability,1011,4.3,STABLE
7,Opportunity,Anime Content Request,872,3.7,FALLING
8,Opportunity,Watch Later Improvements,716,3.0,RISING
9,Opportunity,Streaming Quality Control,599,2.5,RISING


## 12. Final Validation

In [16]:
import json
from pathlib import Path

required = ["data.json", "final_swot_matrix.csv", "theme_evidence.csv",
            "theme_evidence.txt", "theme_assignment_validation.csv"]
for filename in required:
    assert (Path("/kaggle/working") / filename).is_file(), f"Missing output: {filename}"
with Path("/kaggle/working/data.json").open(encoding="utf-8") as f:
    dashboard = json.load(f)

assert dashboard["swotScope"] == "historical"
assert dashboard["meta"]["total_reviews"] == len(df_raw)
assert dashboard["meta"]["total_opinions"] == len(CLASSIFIED)
assert sum(dashboard["ratingCounts"]) == dashboard["meta"]["total_reviews"]
assert 1.0 <= dashboard["meta"]["monthly_avg_rating"] <= 5.0
assert dashboard["meta"]["monthly_rating_review_count"] == len(df_current_month)
assert dashboard["meta"]["monthly_rating_review_count"] == dashboard["monthlyStats"][str(CURRENT_MONTH)]["raw_reviews"]
assert abs(dashboard["meta"]["monthly_avg_rating"] - round(float(df_current_month["rating"].mean()), 2)) < 1e-8
assert abs(dashboard["kpiChanges"]["monthly_rating_vs_historical"] - round(float(df_current_month["rating"].mean() - df_raw["rating"].mean()), 2)) < 1e-8
assert all(
    isinstance(t["insight"], str) and t["insight"].strip() and t["insight"].lower() != "nan"
    for direction in ("rising", "falling") for t in dashboard["momentum"][direction]
)
assert sum(dashboard["opinionTotals"].values()) == dashboard["meta"]["total_opinions"]
assert dashboard["quadTotals"] == QUAD_TOTALS
assert dashboard["monthlyReviewTotals"] == MONTHLY_QUAD_TOTALS
assert dashboard["displayedThemeCount"] == len(final_swot) == len(dashboard["topics"])
assert len({t["id"] for t in dashboard["topics"]}) == len(dashboard["topics"])
assert dashboard["themeCount"] == len(validation_df)
assert dashboard["meta"]["macro_f1"] == round(float(MACRO_F1), 2)
assert "coverage_pct" not in dashboard["meta"]
assert set(dashboard["summary"]) == set(SWOT_LABELS)
assert all(len(dashboard["heatmap"][c]) == 5 for c in SWOT_LABELS)
assert all(len(dashboard["trend"][c]) == len(dashboard["trendQuarters"]) for c in SWOT_LABELS)
assert len(dashboard["volume"]["quarters"]) == len(dashboard["volume"]["reviewCounts"])
assert all(t["mentions"] > 0 and t["quotes"] for t in dashboard["topics"])
for item in dashboard["topics"]:
    cat = item["quad"]
    source = final_swot.loc[
        final_swot["swot"].eq(cat) & final_swot["topic_name"].eq(item["name"])
    ]
    assert len(source) == 1
    assert item["mentions"] == int(source.iloc[0]["historical_mention_count"])
    assert item["mentions"] <= QUAD_TOTALS[cat]
    assigned = theme_assignments[cat].loc[theme_assignments[cat]["topic_name"].eq(item["name"])]
    for rating, text, date in item["quotes"]:
        match = assigned.loc[
            assigned["text_clean"].eq(text) & assigned["rating"].eq(rating) &
            assigned["date"].dt.strftime("%b %Y").eq(date)
        ]
        assert not match.empty, f"Invalid quote provenance: {item['name']}"

print("Dashboard validation: PASSED")
print("SWOT period:", dashboard["period_label"])
print("Reviews:", dashboard["meta"]["total_reviews"])
print("Opinions:", dashboard["meta"]["total_opinions"])
print("Discovered themes:", dashboard["themeCount"])
print("Displayed themes:", dashboard["displayedThemeCount"])

Dashboard validation: PASSED
SWOT period: Jan 2020 – Oct 2026
Reviews: 142474
Opinions: 252290
Discovered themes: 168
Displayed themes: 45
